# Demo 05 — Structured Invoice Extraction

This notebook converts unstructured invoice text into a validated `StructuredInvoice`.

### Objectives

1. Load the document-intelligence result.
2. Verify the extracted-text fingerprint.
3. Inspect the structured-extraction interface.
4. Invoke the invoice extraction agent.
5. Validate the extracted business fields.
6. Display confidence and validation evidence.
7. Save the structured invoice to `demo_session.json`.

## Validate the Unified Invoice Text

Before the AI agent receives the document content, this notebook independently confirms:

- Document intelligence completed successfully.
- The PDF was classified as `TEXT_READY`.
- Embedded-text extraction was used.
- Unified text is present.
- The text SHA-256 fingerprint is unchanged.
- Human review was not required during document extraction.

This prevents incomplete or altered text from entering the structured-extraction agent.

In [6]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import inspect
import json
import os
import sys

PROJECT_ROOT = Path(
    "/Users/pmayank/workspace/LdcDemo"
).resolve()

SESSION_PATH = PROJECT_ROOT / "data" / "demo_session.json"

os.chdir(PROJECT_ROOT)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("=" * 80)
print("LOAD UNIFIED INVOICE TEXT")
print("=" * 80)

assert SESSION_PATH.exists(), (
    "demo_session.json was not found. "
    "Complete Notebook 04 first."
)

demo_session = json.loads(
    SESSION_PATH.read_text(encoding="utf-8")
)

invoice_filename = demo_session["invoice_filename"]
invoice_path = Path(
    demo_session["invoice_path"]
).resolve()

document_hash = demo_session["document_hash"]
case_reference = demo_session["case_reference"]
event_id = demo_session["event_id"]
correlation_id = demo_session["correlation_id"]

document_intelligence = demo_session.get(
    "document_intelligence",
    {},
)

document_profile = document_intelligence.get(
    "profile",
    {},
)

document_extraction = document_intelligence.get(
    "extraction",
    {},
)

unified_text = document_extraction.get(
    "unified_text",
    "",
)

stored_text_hash = document_extraction.get(
    "unified_text_sha256"
)

calculated_text_hash = hashlib.sha256(
    unified_text.encode("utf-8")
).hexdigest()

stored_character_count = document_extraction.get(
    "total_character_count",
    0,
)

text_validation = {
    "Session status is DOCUMENT_INTELLIGENCE_COMPLETED": (
        demo_session.get("session_status")
        == "DOCUMENT_INTELLIGENCE_COMPLETED"
    ),
    "Document-intelligence chapter is complete": (
        "04_DOCUMENT_INTELLIGENCE"
        in demo_session.get("completed_chapters", [])
    ),
    "Document profile was verified": (
        document_intelligence.get("profile_verified")
        is True
    ),
    "Document extraction was verified": (
        document_intelligence.get("extraction_verified")
        is True
    ),
    "Profile outcome is TEXT_READY": (
        document_profile.get("quality_status")
        == "TEXT_READY"
    ),
    "Embedded-text route was used": (
        document_extraction.get("extractor_node_id")
        == "extract_embedded_text"
    ),
    "Unified text is available": (
        isinstance(unified_text, str)
        and bool(unified_text.strip())
    ),
    "Text fingerprint is unchanged": (
        calculated_text_hash == stored_text_hash
    ),
   "Character count is valid": (
    len(unified_text) > 0
    and stored_character_count > 0
    ),
    "Document extraction needs no human review": (
        document_extraction.get(
            "requires_human_review"
        )
        is False
    ),
}

print(f"Invoice          : {invoice_filename}")
print(f"Case reference   : {case_reference}")
print(f"Correlation ID   : {correlation_id}")
print(f"Text characters  : {len(unified_text):,}")
print(f"Stored text hash : {stored_text_hash}")
print(f"Current text hash: {calculated_text_hash}")
print("-" * 80)

for check_name, passed in text_validation.items():
    symbol = "✅" if passed else "❌"
    status = "PASS" if passed else "FAIL"
    print(f"{symbol} {status:<4} | {check_name}")

unified_text_ready = all(
    text_validation.values()
)

print("-" * 80)

if unified_text_ready:
    print("✅ UNIFIED INVOICE TEXT VERIFIED")
    print("The content is ready for structured extraction.")
else:
    print("❌ UNIFIED INVOICE TEXT VALIDATION FAILED")

assert unified_text_ready, (
    "Structured extraction stopped because the unified "
    "invoice text did not pass validation."
)

LOAD UNIFIED INVOICE TEXT
Invoice          : 01_Veson_Bunker_Clean_STP.pdf
Case reference   : LOCAL-B970E86BB68E
Correlation ID   : CORR-2CBC6CC15D7249CB8EE549B035F87E41
Text characters  : 1,347
Stored text hash : 881b0ec294af66302e2f2abf16ac853ad6a60a8bebd2ec74d806a7c2a3985d06
Current text hash: 881b0ec294af66302e2f2abf16ac853ad6a60a8bebd2ec74d806a7c2a3985d06
--------------------------------------------------------------------------------
✅ PASS | Session status is DOCUMENT_INTELLIGENCE_COMPLETED
✅ PASS | Document-intelligence chapter is complete
✅ PASS | Document profile was verified
✅ PASS | Document extraction was verified
✅ PASS | Profile outcome is TEXT_READY
✅ PASS | Embedded-text route was used
✅ PASS | Unified text is available
✅ PASS | Text fingerprint is unchanged
✅ PASS | Character count is valid
✅ PASS | Document extraction needs no human review
--------------------------------------------------------------------------------
✅ UNIFIED INVOICE TEXT VERIFIED
The content is r

In [7]:
from pathlib import Path
import json

PROJECT_ROOT = Path("/Users/pmayank/workspace/LdcDemo")
SESSION_PATH = PROJECT_ROOT / "data"/"demo_session.json"

print("=" * 90)
print("UNIFIED TEXT — DIAGNOSTIC")
print("=" * 90)

assert SESSION_PATH.exists(), f"Session file not found: {SESSION_PATH}"

with SESSION_PATH.open("r", encoding="utf-8") as file:
    session = json.load(file)

print(f"Session file : {SESSION_PATH}")
print(f"Top-level keys: {sorted(session.keys())}")

candidate_paths = {
    "unified_text":
        session.get("unified_text"),

    "extracted_text":
        session.get("extracted_text"),

    "document_intelligence.unified_text":
        session.get("document_intelligence", {}).get("unified_text"),

    "document_intelligence.extracted_text":
        session.get("document_intelligence", {}).get("extracted_text"),

    "document_extraction.unified_text":
        session.get("document_extraction", {}).get("unified_text"),

    "document_extraction.extracted_text":
        session.get("document_extraction", {}).get("extracted_text"),

    "extraction_result.extracted_text":
        session.get("extraction_result", {}).get("extracted_text"),

    "extraction_result.text":
        session.get("extraction_result", {}).get("text"),
}

print("\nCandidate text fields:")

for field_path, value in candidate_paths.items():
    if isinstance(value, str):
        print(
            f"  {field_path:<50} "
            f"type=str | characters={len(value.strip())}"
        )
    elif value is None:
        print(f"  {field_path:<50} missing")
    else:
        print(
            f"  {field_path:<50} "
            f"type={type(value).__name__}"
        )

print("\nNested dictionaries:")

for key, value in session.items():
    if isinstance(value, dict):
        print(f"  {key}: {sorted(value.keys())}")

UNIFIED TEXT — DIAGNOSTIC
Session file : /Users/pmayank/workspace/LdcDemo/data/demo_session.json
Top-level keys: ['baseline_captured_at', 'before_processing_counts', 'case_mode', 'case_reference', 'completed_chapters', 'correlation_id', 'database_path', 'document_hash', 'document_hash_algorithm', 'document_intelligence', 'document_intelligence_completed_at', 'event_id', 'intake', 'intake_completed_at', 'invoice_filename', 'invoice_path', 'invoice_size_bytes', 'pdf_is_valid', 'pdf_page_count', 'prior_evidence_found', 'project_root', 'session_created_at', 'session_status', 'session_updated_at', 'session_version']

Candidate text fields:
  unified_text                                       missing
  extracted_text                                     missing
  document_intelligence.unified_text                 missing
  document_intelligence.extracted_text               missing
  document_extraction.unified_text                   missing
  document_extraction.extracted_text                

## Structured Extraction Contract

The extraction layer has two responsibilities:

1. The extraction agent interprets the verified invoice text.
2. The invoice schema validates the agent's output.

The schema ensures that fields such as dates, currency, amounts and confidence conform to the expected business contract.

In [8]:
import inspect

import src.invoice_extraction_agent as extraction_agent_module
import src.invoice_schema as invoice_schema_module


def discover_module_interfaces(module):
    """Return locally implemented public functions and classes."""
    interfaces = []

    for name, function_object in inspect.getmembers(
        module,
        inspect.isfunction,
    ):
        if name.startswith("_"):
            continue

        if function_object.__module__ != module.__name__:
            continue

        try:
            signature = inspect.signature(function_object)
        except (TypeError, ValueError):
            signature = "Signature unavailable"

        async_label = (
            "async "
            if inspect.iscoroutinefunction(function_object)
            else ""
        )

        interfaces.append(
            {
                "type": "function",
                "name": name,
                "display": f"{async_label}{name}{signature}",
                "object": function_object,
            }
        )

    for name, class_object in inspect.getmembers(
        module,
        inspect.isclass,
    ):
        if class_object.__module__ != module.__name__:
            continue

        try:
            signature = inspect.signature(class_object)
        except (TypeError, ValueError):
            signature = "Signature unavailable"

        interfaces.append(
            {
                "type": "class",
                "name": name,
                "display": f"CLASS {name}{signature}",
                "object": class_object,
            }
        )

        for method_name, method_object in inspect.getmembers(
            class_object,
            inspect.isfunction,
        ):
            if method_name.startswith("_"):
                continue

            try:
                method_signature = inspect.signature(
                    method_object
                )
            except (TypeError, ValueError):
                method_signature = "Signature unavailable"

            async_label = (
                "async "
                if inspect.iscoroutinefunction(method_object)
                else ""
            )

            interfaces.append(
                {
                    "type": "method",
                    "name": f"{name}.{method_name}",
                    "display": (
                        f"  {async_label}{method_name}"
                        f"{method_signature}"
                    ),
                    "object": method_object,
                }
            )

    return interfaces


print("=" * 100)
print("STRUCTURED INVOICE EXTRACTION AGENT")
print("=" * 100)
print(f"Module : {extraction_agent_module.__name__}")
print(f"File   : {inspect.getfile(extraction_agent_module)}")
print("-" * 100)

agent_interfaces = discover_module_interfaces(
    extraction_agent_module
)

for interface in agent_interfaces:
    print(interface["display"])

print("\n" + "=" * 100)
print("STRUCTURED INVOICE SCHEMA")
print("=" * 100)
print(f"Module : {invoice_schema_module.__name__}")
print(f"File   : {inspect.getfile(invoice_schema_module)}")
print("-" * 100)

schema_interfaces = discover_module_interfaces(
    invoice_schema_module
)

for interface in schema_interfaces:
    print(interface["display"])

print("\n" + "=" * 100)
print("PYDANTIC SCHEMA FIELDS")
print("=" * 100)

pydantic_models_found = []

for class_name, class_object in inspect.getmembers(
    invoice_schema_module,
    inspect.isclass,
):
    if class_object.__module__ != invoice_schema_module.__name__:
        continue

    model_fields = getattr(
        class_object,
        "model_fields",
        None,
    )

    if not model_fields:
        continue

    pydantic_models_found.append(class_name)

    print(f"\nMODEL: {class_name}")

    for field_name, field_info in model_fields.items():
        annotation = field_info.annotation

        required = (
            field_info.is_required()
            if callable(
                getattr(field_info, "is_required", None)
            )
            else "Unknown"
        )

        print(
            f"  {field_name:<30} "
            f"type={annotation!s:<35} "
            f"required={required}"
        )

interface_discovery_ready = (
    bool(agent_interfaces)
    and bool(schema_interfaces)
    and bool(pydantic_models_found)
)

print("\n" + "-" * 100)

if interface_discovery_ready:
    print("✅ EXTRACTION AGENT INTERFACE DISCOVERED")
    print("✅ STRUCTURED INVOICE SCHEMA DISCOVERED")
else:
    print("❌ STRUCTURED EXTRACTION INTERFACE IS INCOMPLETE")

assert interface_discovery_ready, (
    "The structured extraction interfaces could not be "
    "discovered completely."
)

STRUCTURED INVOICE EXTRACTION AGENT
Module : src.invoice_extraction_agent
File   : /Users/pmayank/workspace/LdcDemo/src/invoice_extraction_agent.py
----------------------------------------------------------------------------------------------------
build_extraction_signature() -> kaizen.signatures.core.Signature
build_invoice_extraction_workflow()
extract_invoice_fields(extracted_text: str, extraction_method: str, ocr_confidence: Optional[str] = None, source_file: str = 'unknown', workflow_run_id: str = '') -> Dict[str, Any]
extract_invoice_fields_via_kaizen(extracted_text: str, extraction_method: str, ocr_confidence: Optional[str] = None, source_file: str = 'unknown') -> Tuple[Dict[str, Any], str]
kaizen_extraction_handler(extracted_text: str, extraction_method: str, ocr_confidence: str, source_file: str) -> Dict[str, Any]
validate_structured_invoice(invoice_dict: Dict[str, Any]) -> Tuple[Optional[src.invoice_schema.StructuredInvoice], list[str]]
CLASS ExtractionResult(structured_invo

## Invoke the Structured Extraction Agent

The verified invoice text is submitted to the Kaizen extraction workflow using Anthropic Claude Haiku 4.5.

The agent must return structured invoice fields, but its output is not trusted automatically. The response is subsequently validated against the `StructuredInvoice` Pydantic schema.

In [ ]:
from dotenv import load_dotenv
from pprint import pprint
import inspect
import os

from src.invoice_extraction_agent import (
    extract_invoice_fields,
    validate_structured_invoice,
)

# Load local environment variables without overwriting
# variables already present in the notebook environment.
load_dotenv(PROJECT_ROOT / ".env", override=False)

anthropic_api_key_present = bool(
    os.getenv("ANTHROPIC_API_KEY")
)
print(f"ANTHROPIC_API_KEY present: {os.getenv('ANTHROPIC_API_KEY')}")
EXPECTED_MODEL = "claude-haiku-4-5"
EXPECTED_PROMPT_VERSION = "invoice-extraction-v1"

print("=" * 90)
print("ANTHROPIC PROVIDER READINESS")
print("=" * 90)
print(f"API key available : {anthropic_api_key_present}")
print(f"Expected model    : {EXPECTED_MODEL}")
print(f"Prompt version    : {EXPECTED_PROMPT_VERSION}")
print("-" * 90)

assert anthropic_api_key_present, (
    "ANTHROPIC_API_KEY is unavailable. Add it to the project "
    ".env file or export it in the terminal before continuing."
)

print("✅ ANTHROPIC PROVIDER CONFIGURATION AVAILABLE")


async def resolve_result(value):
    """Support either synchronous or asynchronous implementation."""
    if inspect.isawaitable(value):
        return await value
    return value


structured_extraction_run_id = (
    f"STRUCT-{correlation_id}"
)

source_extraction_method = (
    document_extraction.get("extraction_method")
    or "embedded_text"
)

source_ocr_confidence = (
    document_extraction.get("ocr_avg_confidence")
)

if source_ocr_confidence is not None:
    source_ocr_confidence = str(
        source_ocr_confidence
    )

print("\n" + "=" * 90)
print("STRUCTURED INVOICE EXTRACTION")
print("=" * 90)
print(f"Invoice          : {invoice_filename}")
print(f"Case reference   : {case_reference}")
print(f"Workflow run ID  : {structured_extraction_run_id}")
print(f"Extraction method: {source_extraction_method}")
print(f"Input characters : {len(unified_text):,}")
print("-" * 90)
print("Invoking the Kaizen/Anthropic extraction agent...")

raw_structured_result = await resolve_result(
    extract_invoice_fields(
        extracted_text=unified_text,
        extraction_method=source_extraction_method,
        ocr_confidence=source_ocr_confidence,
        source_file=invoice_filename,
        workflow_run_id=structured_extraction_run_id,
    )
)

assert isinstance(raw_structured_result, dict), (
    "The extraction agent did not return a dictionary."
)

print("✅ EXTRACTION AGENT RETURNED A RESPONSE")
print(
    "Response keys: "
    + ", ".join(sorted(raw_structured_result.keys()))
)

[NODE] Unknown parameter(s) for HandlerNode: ['handler', 'params']. Valid parameters: ['extracted_text', 'extraction_method', 'ocr_confidence', 'source_file'].
[NODE] Unknown parameter(s) for HandlerNode: ['handler', 'params']. Valid parameters: ['extracted_text', 'extraction_method', 'ocr_confidence', 'source_file'].


ANTHROPIC_API_KEY present: True
ANTHROPIC PROVIDER READINESS
API key available : True
Expected model    : claude-haiku-4-5
Prompt version    : invoice-extraction-v1
------------------------------------------------------------------------------------------
✅ ANTHROPIC PROVIDER CONFIGURATION AVAILABLE

STRUCTURED INVOICE EXTRACTION
Invoice          : 01_Veson_Bunker_Clean_STP.pdf
Case reference   : LOCAL-B970E86BB68E
Workflow run ID  : STRUCT-CORR-2CBC6CC15D7249CB8EE549B035F87E41
Extraction method: EMBEDDED_TEXT
Input characters : 1,347
------------------------------------------------------------------------------------------
Invoking the Kaizen/Anthropic extraction agent...


Provider anthropic error [ProviderError]: anthropic error (ProviderError): provider error: status=400 body='{"type":"error","error":{"type":"invalid_request_error","message":"Your credit balance is too low to access the Anthropic API. Please go to Plans & Billing to upgrade or purchase credits."},"request_id":"req_011Cf3aZ479KJGJzpuf9hxQS"}'
Node invoice_extractor execution failed: anthropic error (ProviderError): provider error: status=400 body='{"type":"error","error":{"type":"invalid_request_error","message":"Your credit balance is too low to access the Anthropic API. Please go to Plans & Billing to upgrade or purchase credits."},"request_id":"req_011Cf3aZ479KJGJzpuf9hxQS"}'
Traceback (most recent call last):
  File "/Users/pmayank/workspace/LdcDemo/.venv/lib/python3.12/site-packages/kailash/runtime/local.py", line 1261, in execute
    loop = asyncio.get_running_loop()
           ^^^^^^^^^^^^^^^^^^^^^^^^^^
RuntimeError: no running event loop

During handling of the above exception, 

✅ EXTRACTION AGENT RETURNED A RESPONSE
Response keys: confidence, model, prompt_version, provider, structured_invoice, validation_failed, warnings, workflow_run_id
